# T2 BLIP Base — Flickr30k pseudo-caption pilot

Pilot này là gate bắt buộc trước full run. Ba pilot T1, T2 và T3 độc lập và có
thể chạy song song trong ba phiên Kaggle. Không đọc Validation caption và không
mở Held-out Test.

## Kaggle input bắt buộc — đúng 6 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_T2_Handoff`.
4. Output `phase3_teacher_b_seed42`.
5. Dataset `Flickr Image dataset`.
6. Dataset/Kaggle Model BLIP Base offline có `MODEL_SOURCE_MANIFEST.json`.

## Runtime

- Accelerator: GPU T4; một GPU là đủ.
- Internet: Off.
- T1/T2: 128 ảnh; T3: 100 optimizer-step.
- Pilot T3 bị hủy trọng số sau đo thời gian; full run phải khởi tạo lại.


## Cell 1 — Input gate, Flickr30k contract và image index

In [1]:

import os
EXPERIMENT_ID = "F30K_T2_Pseudo_Pilot"
TEACHER_ID = "T2"
HANDOFF_RESULT_NAME = "F30K_T2_Handoff_RESULT.json"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import csv, hashlib, json, math, os, random, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def image_index(required_names):
    candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in required_names]
    by_name = {}
    duplicate_aliases = 0
    for path in candidates:
        old = by_name.get(path.name)
        if old is None:
            by_name[path.name] = path
            continue
        if old.resolve() == path.resolve():
            continue
        assert old.stat().st_size == path.stat().st_size, (
            "Ảnh trùng tên nhưng khác nội dung/kích thước", path.name, old, path
        )
        duplicate_aliases += 1
        by_name[path.name] = min(
            (old, path), key=lambda p: (len(p.parts), len(str(p)), str(p))
        )
    missing = sorted(required_names - set(by_name))
    assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])
    return by_name, duplicate_aliases

DATA_PATH = unique_named("f30k_train.parquet")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")
HANDOFF_PATH = unique_named(HANDOFF_RESULT_NAME)

data_contract = read_json(DATA_CONTRACT_PATH)
base_ready = read_json(BASE_READY_PATH)
vocab = read_json(VOCAB_PATH)
handoff = read_json(HANDOFF_PATH)

assert data_contract["status"] == "complete"
assert data_contract["train_image_count"] == 29000
assert data_contract["validation_image_count"] == 1014
assert data_contract["test_image_count"] == 1000
assert data_contract["vocabulary_size"] == 7456
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False
assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert handoff["teacher_id"] == TEACHER_ID
assert handoff["flickr30k"]["test_set_status"] == "sealed"
assert handoff["flickr30k"]["test_set_used"] is False
assert sha256(DATA_PATH) == data_contract["artifacts"]["train"]["sha256"]
assert sha256(VOCAB_PATH) == data_contract["artifacts"]["vocabulary"]["sha256"]

train_df = pd.read_parquet(DATA_PATH)
assert train_df["image_id"].nunique() == 29000
assert len(train_df) == 145000
train_ids = sorted(int(x) for x in train_df["image_id"].unique())
required_names = {f"{image_id}.jpg" for image_id in train_ids}
IMAGE_BY_NAME, DUPLICATE_ALIASES = image_index(required_names)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
assert torch.cuda.is_available(), "Notebook pilot yêu cầu GPU T4."
DEVICE = torch.device("cuda:0")

print("Common gate             : PASSED")
print("Experiment              :", EXPERIMENT_ID)
print("Teacher                 :", TEACHER_ID)
print("Flickr30k Train         : 29,000 images")
print("Vocabulary              : 7,456")
print("Nested image aliases    :", DUPLICATE_ALIASES, "(deduplicated)")
print("Held-out Test           : SEALED / NOT READ")


Common gate             : PASSED
Experiment              : F30K_T2_Pseudo_Pilot
Teacher                 : T2
Flickr30k Train         : 29,000 images
Vocabulary              : 7,456
Nested image aliases    : 29000 (deduplicated)
Held-out Test           : SEALED / NOT READ


## Cell 2 — Teacher workload pilot

In [2]:

from torch.utils.data import DataLoader, Dataset
from transformers import BlipForConditionalGeneration, BlipProcessor

assert handoff["status"] == "passed"
assert handoff["method_contract"]["selected_method"] == "Sequence KD"
EXPECTED_CHECKPOINT_SHA = handoff["teacher_source"]["development_checkpoint_sha256"]
CHECKPOINT_PATH = unique_named("teacher_b_seed42_best.pt")
assert sha256(CHECKPOINT_PATH) == EXPECTED_CHECKPOINT_SHA

model_sources = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(manifest_path)
    except Exception:
        continue
    if (
        payload.get("repo_id") == "Salesforce/blip-image-captioning-base"
        and payload.get("resolved_commit") == "82a37760796d32b1411fe092ab5d4e227313294b"
    ):
        model_sources.append(manifest_path.parent)
assert len(model_sources) == 1, model_sources
MODEL_SOURCE = model_sources[0]

processor = BlipProcessor.from_pretrained(
    str(MODEL_SOURCE), local_files_only=True, use_fast=False
)
tokenizer = processor.tokenizer
if "[DEC]" not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"bos_token": "[DEC]"})
else:
    tokenizer.bos_token = "[DEC]"
if "[ENC]" not in tokenizer.get_vocab():
    current = tokenizer.special_tokens_map.get("additional_special_tokens", [])
    if isinstance(current, str): current = [current]
    tokenizer.add_special_tokens({"additional_special_tokens": list(current) + ["[ENC]"]})
assert tokenizer.convert_tokens_to_ids("[DEC]") == 30522
assert tokenizer.convert_tokens_to_ids("[ENC]") == 30523

model = BlipForConditionalGeneration.from_pretrained(
    str(MODEL_SOURCE), local_files_only=True
)
payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
assert payload["schema_version"] == "teacher_b_seed42_best_model_v1"
assert payload["seed"] == 42 and payload["test_set_used"] is False
load_result = model.load_state_dict(payload["model"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
model = model.to(DEVICE).eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

PILOT_IDS = train_ids[:128]
class PilotImages(Dataset):
    def __len__(self): return len(PILOT_IDS)
    def __getitem__(self, index):
        image_id = PILOT_IDS[index]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as image:
            return image_id, image.convert("RGB").copy()

def collate(rows):
    ids, images = zip(*rows)
    values = processor(images=list(images), return_tensors="pt")["pixel_values"]
    return torch.tensor(ids), values

loader = DataLoader(PilotImages(), batch_size=8, shuffle=False, num_workers=2, collate_fn=collate)
records = []
started = time.perf_counter()
torch.cuda.reset_peak_memory_stats()
for image_ids, pixels in loader:
    pixels = pixels.to(DEVICE, non_blocking=True)
    with torch.inference_mode():
        sequences = model.generate(
            pixel_values=pixels, num_beams=3, do_sample=False,
            length_penalty=0.7, max_new_tokens=31, min_new_tokens=2,
            early_stopping=True,
        )
    captions = [x.strip() for x in processor.batch_decode(sequences, skip_special_tokens=True)]
    for image_id, token_ids, caption in zip(
        image_ids.tolist(), sequences.detach().cpu().tolist(), captions
    ):
        assert caption
        records.append({
            "image_id": int(image_id), "caption": caption,
            "teacher_token_ids": [int(x) for x in token_ids],
        })
elapsed = time.perf_counter() - started
assert len(records) == 128 and len({x["image_id"] for x in records}) == 128
rate = len(records) / elapsed
eta_hours = 29000 / rate / 3600
(WORK_ROOT / "pilot_captions.json").write_text(
    json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8"
)
RESULT = {
    "schema_version": "f30k_t2_pseudo_pilot_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed", "pilot_gate": "full_pseudocaption_generation_authorized",
    "teacher": "BLIP Base", "matched_student": "S2 MobileViT-S",
    "pilot_images": 128, "images_per_second": rate,
    "estimated_full_generation_hours": eta_hours,
    "checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "beam_size": 3, "length_penalty": 0.7, "maximum_new_tokens": 31,
    "next_phase_authorized": "F30K_T2_Pseudo_Full",
    "test_set_status": "sealed", "test_set_used": False,
}
print("Pilot captions          : 128 / 128")
print("Images/second           :", round(rate, 4))
print("Estimated full runtime  :", round(eta_hours, 2), "hours")


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Pilot captions          : 128 / 128
Images/second           : 6.6971
Estimated full runtime  : 1.2 hours


## Cell 3 — Result và manifest SHA-256

In [3]:

RESULT_PATH = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(RESULT_PATH, RESULT)
artifacts = []
for path in sorted(p for p in WORK_ROOT.rglob("*") if p.is_file() and p.name != "MANIFEST.json"):
    artifacts.append({
        "relative_path": path.relative_to(WORK_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
MANIFEST_PATH = WORK_ROOT / "MANIFEST.json"
write_json(
    MANIFEST_PATH,
    {
        "schema_version": "f30k_teacher_workload_pilot_manifest_v1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": EXPERIMENT_ID,
        "status": RESULT["status"],
        "artifacts": artifacts,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
print("=" * 78)
print(EXPERIMENT_ID, "— FINAL")
print("=" * 78)
print("Status                  :", RESULT["status"])
print("Pilot gate              :", RESULT["pilot_gate"])
print("Next                    :", RESULT["next_phase_authorized"])
print("Held-out Test           : SEALED / NOT READ")
print("Output                  :", WORK_ROOT)


F30K_T2_Pseudo_Pilot — FINAL
Status                  : passed
Pilot gate              : full_pseudocaption_generation_authorized
Next                    : F30K_T2_Pseudo_Full
Held-out Test           : SEALED / NOT READ
Output                  : /kaggle/working/F30K_T2_Pseudo_Pilot
